In [ ]:
%load_ext autoreload
%autoreload 2

from io import StringIO
import re
import time

import pandas as pd
from pathlib import Path
from Bio import SeqIO
from notebooks.jacob.round3.util import reindex_circular_genbank, find_mutations, Mutation
from app.helpers.sequence_util import allele_set_to_seq_id, maybe_get_allele_id_error_message

def get_results(reference_path, genbank_dir):
    # Load reference
    reference = SeqIO.read(reference_path, 'genbank')
    print(f"Loaded reference: {reference.id}, length={len(reference.seq)}")

    results = {}

    genbank_files = sorted(genbank_dir.glob('*.gbk'))
    print(f"Found {len(genbank_files)} genbank files to process\n")

    for gbk_file in genbank_files:
        print(f"Processing {gbk_file.name}...", end=' ')
        start_time = time.time()
        
        try:
            # Load query sequence
            query = SeqIO.read(gbk_file, 'genbank')
            
            # Reindex to match reference
            reindexed_query = reindex_circular_genbank(reference, query)
            
            # Find mutations
            mutations = find_mutations(reference, reindexed_query)
            
            # Store results
            results[gbk_file.name] = mutations
            
            elapsed = time.time() - start_time
            print(f"found {len(mutations)} mutations ({len(mutations)} grouped) ({elapsed:.2f}s)")
            
        except Exception as e:
            print(f"ERROR: {e}")
            import traceback
            traceback.print_exc()
            results[gbk_file.name] = None
    return results

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
oplr_reference_path = Path('notebooks/jacob/round1/template_plasmids/real_oplr_wt.gb')
oplr_genbank_dir = Path('notebooks/jacob/round3/251013_oplr_r2_reverify_genbank-files')
oplr_results = get_results(oplr_reference_path, oplr_genbank_dir)   


Loaded reference: pAP_OplR_CH_R3_0001, length=6049
Found 43 genbank files to process

Processing CZNHZP_10_oplr_r2_plate.B10.gbk... found 2 mutations (2 grouped) (0.11s)
Processing CZNHZP_11_oplr_r2_plate.B11.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_12_oplr_r2_plate.B12.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_1_oplr_r2_plate.B1.gbk... found 3 mutations (3 grouped) (0.10s)
Processing CZNHZP_2_oplr_r2_plate.B2.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_3_oplr_r2_plate.B3.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_4_oplr_r2_plate.B4.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_5_oplr_r2_plate.B5.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_6_oplr_r2_plate.B6.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_7_oplr_r2_plate.B7.gbk... found 2 mutations (2 grouped) (0.09s)
Processing CZNHZP_8_oplr_r2_plate.B8.gbk... found 2 mutations (2 grouped) (0.09s)
Proces

In [26]:
discovered_mut_list = []
for filename, mut_list in oplr_results.items():
    sequence_well = filename.split('.')[1]
    padded_sequence_well = f'{sequence_well[0]}{int(sequence_well[1:]):02d}'

    # Extract allele set from CDS mutations (excluding ? mutations)
    cds_mutations = [m for m in mut_list if m.amino_acid_change is not None]

    allele_set = set()
    for mut in cds_mutations:
        aa_change = mut.amino_acid_change
        # If its a synonymous mutation, skip it.
        if re.match(r'[A-Z]\d+[A-Z]', aa_change) and aa_change[0] == aa_change[-1]:
            continue
        # if aa_change and '?' not in aa_change and 'frameshift' not in aa_change.lower():
        allele_set.add(aa_change)
    try:
        found_seq_id = allele_set_to_seq_id(allele_set)
    except Exception as e:
        found_seq_id = '_'.join(allele_set)
    
    discovered_mut_list.append({
        'sequence_well': sequence_well,
        'padded_sequence_well': padded_sequence_well,
        'discovered_mutations': found_seq_id,
    })
    
discovered_mut_df = pd.DataFrame(discovered_mut_list).sort_values(by='padded_sequence_well')
    

In [28]:
for ii in range(len(discovered_mut_df)):
    print(f'{discovered_mut_df["discovered_mutations"].iloc[ii]}\t{discovered_mut_df["sequence_well"].iloc[ii]}')
    
    


A3V_Q123D_Y255R	B1
Q123D_Q264K	B2
Q123D_Q296H	B3
Q123D_Q303R	B4
Q123D_A309P	B5
K137A_Y255L	B6
K137P_Y255L	B7
K144A_Y255L	B8
K144D_Y255L	B9
K144D_Y255L	B10
Y146A_Y255T	B11
Q123D_A309P	B12
Y146I_Y255T	C1
Y146L_Y255T	C2
Y255L_A309P	C3
E218L_Y255T	C4
E218R_Y255L	C5
E218R_Y255M	C6
E218R_Y255T	C7
Y220H_Y255M	C8
Y220H_Y255T	C9
E218I_Y255T	C11
E218R_Y255M	C12
Y220H_Y255M	D1
Y220H_Y255T	D2
K144E_Y255L_E311*	D4
Y146M_Y255T	D5
Y146T_Y255T	D6
F80A	D7
Y255L	D8
WT	D9
Q12D_Q123D	E1
Q45S_Q123D_A292D	E2
T52G_Y255T	E3
Q63A_E218I	E4
Q63V_Q123D	E5
Q75R_Y255T	E6
F80A_Y255R	E7
Q99R_Q123D	E8
Q123D_Q155S	E9
G11D_Q123D_G237K	E10
Q123D_L241I	E11
Q123D_Y255R	E12
